In [1]:
# train.py
import os, re, json, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.graph_objects as go 
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

EMOTION_COLUMNS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

DATA_PATH = "go_emotions_dataset.csv"
MODEL_NAME = "distilbert-base-uncased"
MODEL_SAVE_DIR = "./goemotions_model_v3"
EVAL_DIR = "./eval_outputs"  
os.makedirs(EVAL_DIR, exist_ok=True)

SAMPLE_SIZE = 15000
MIN_MAX_LEN = 16
MAX_MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 3

NUM_LABELS = len(EMOTION_COLUMNS)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

PLOTLY_FONT_COLOR = "#CBD5E1"
PLOTLY_TITLE_COLOR = "#E2E8F0"
PLOTLY_GRID_COLOR = "#262B45"
PLOTLY_ZERO_COLOR = "#64748B"


def load_dataset(path):
    df = pd.read_csv(path)
    if "example_very_unclear" in df.columns:
        df = df[df["example_very_unclear"] == False]  # noqa: E712
    df = df[df[EMOTION_COLUMNS].sum(axis=1) > 0].reset_index(drop=True)
    for col in EMOTION_COLUMNS:
        df[col] = df[col].astype(int)
    return df


def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def analyze_token_lengths(texts, tokenizer, save_path_png, save_path_json):
    """
    Computes real token-length stats (no truncation) so MAX_LEN is a
    justified choice instead of a guess.

    CHANGE: now saves TWO versions of the histogram —
      - save_path_json: an interactive Plotly figure (fig.write_json), which
        is what app.py's "How It Works" tab loads via plotly.io.read_json()
        and renders with gr.Plot() so hovering shows exact bin counts/values.
      - save_path_png: the original static matplotlib image, kept only as a
        fallback in case app.py is ever run before this script produces the
        JSON, or by any other tooling that still expects a PNG on disk.
    """
    lengths = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in texts]
    lengths = np.array(lengths)
    percentiles = {p: int(np.percentile(lengths, p)) for p in (50, 90, 95, 99)}
    print("Token length percentiles:", percentiles,
          f"| max observed: {lengths.max()}")

    # ---- interactive Plotly version (primary — consumed by app.py) ----
    fig = go.Figure(go.Histogram(
        x=lengths, nbinsx=40, marker_color="#4C72B0",
        hovertemplate="Token length: %{x}<br>Count: %{y}<extra></extra>",
    ))
    for p, v in percentiles.items():
        fig.add_vline(
            x=v, line_dash="dash", line_width=1, line_color="#C44E52",
            annotation_text=f"p{p}={v}", annotation_position="top",
            annotation_textangle=-90,
            annotation_font=dict(color="#FCA5A5", size=10),
        )
    fig.update_layout(
        title=dict(text="Token length distribution (justifies MAX_LEN choice)",
                   font=dict(color=PLOTLY_TITLE_COLOR, size=14)),
        xaxis_title="Token length (with special tokens)",
        yaxis_title="Count",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color=PLOTLY_FONT_COLOR),
        height=420,
        margin=dict(l=10, r=10, t=50, b=10),
        bargap=0.02,
    )
    fig.update_xaxes(gridcolor=PLOTLY_GRID_COLOR, zerolinecolor=PLOTLY_ZERO_COLOR, color=PLOTLY_FONT_COLOR)
    fig.update_yaxes(gridcolor=PLOTLY_GRID_COLOR, zerolinecolor=PLOTLY_ZERO_COLOR, color=PLOTLY_FONT_COLOR)
    fig.write_json(save_path_json)

    # ---- static matplotlib fallback (unchanged from before, kept for safety) ----
    plt.figure(figsize=(7, 4))
    plt.hist(lengths, bins=40, color="#4C72B0")
    for p, v in percentiles.items():
        plt.axvline(v, linestyle="--", linewidth=1, color="#C44E52")
        plt.text(v, plt.ylim()[1] * 0.9, f"p{p}={v}", rotation=90, fontsize=8)
    plt.xlabel("Token length (with special tokens)")
    plt.ylabel("Count")
    plt.title("Token length distribution (justifies MAX_LEN choice)")
    plt.tight_layout()
    plt.savefig(save_path_png, dpi=120)
    plt.close()

    chosen = int(np.clip(percentiles[95], MIN_MAX_LEN, MAX_MAX_LEN))
    print(f"Chosen MAX_LEN = {chosen} (95th percentile, clipped to "
          f"[{MIN_MAX_LEN}, {MAX_MAX_LEN}])")
    return chosen, percentiles


class GoEmotionsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length",
                              max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="micro", zero_division=0),
        "recall": recall_score(labels, preds, average="micro", zero_division=0),
        "subset_accuracy": accuracy_score(labels, preds),
    }


class WeightedTrainer(Trainer):
    """
    Plain BCE loss lets 'neutral' dominate since it appears in ~30% of rows
    while emotions like 'gratitude' appear in <2%. pos_weight rebalances
    the loss so rare labels aren't ignored.
    """
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def main():
    start = time.time()
    df = load_dataset(DATA_PATH)
    full_dataset_size = len(df)
    print(f"Loaded {full_dataset_size} usable rows")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification"
    ).to(DEVICE)

    actual_sample_size = min(SAMPLE_SIZE, full_dataset_size)
    sample_fraction = actual_sample_size / full_dataset_size
    print(f"Training on {actual_sample_size}/{full_dataset_size} rows "
          f"({sample_fraction:.1%} of the full dataset) — a compute tradeoff, "
          f"documented in metrics.json as a limitation.")

    sample = df.sample(n=actual_sample_size, random_state=42).reset_index(drop=True)
    texts = sample["text"].apply(clean_text).tolist()
    labels = sample[EMOTION_COLUMNS].values.tolist()

    # ---- justify MAX_LEN with real data instead of guessing ----
    MAX_LEN, length_percentiles = analyze_token_lengths(
        texts, tokenizer,
        save_path_png=os.path.join(EVAL_DIR, "token_length_hist.png"),
        save_path_json=os.path.join(EVAL_DIR, "token_length_hist.json"),
    )

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.15, random_state=42
    )

    train_labels_arr = np.array(train_labels)
    pos_counts = train_labels_arr.sum(axis=0)
    neg_counts = len(train_labels_arr) - pos_counts
    pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 15.0)
    pos_weight = torch.tensor(pos_weight, dtype=torch.float)
    print("Rarest labels (highest weight given):",
          [EMOTION_COLUMNS[i] for i in np.argsort(-pos_weight.numpy())[:5]])

    train_ds = GoEmotionsDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
    val_ds = GoEmotionsDataset(val_texts, val_labels, tokenizer, max_len=MAX_LEN)

    common_args = dict(
        output_dir="./checkpoints3",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        save_strategy="epoch",
        logging_steps=5,
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        learning_rate=2e-5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
        report_to=[],
    )
    try:
        args = TrainingArguments(eval_strategy="epoch", **common_args)
    except TypeError:
        args = TrainingArguments(evaluation_strategy="epoch", **common_args)

    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics, pos_weight=pos_weight,
    )
    trainer.train()

    eval_metrics = trainer.evaluate()
    print("\nFinal validation metrics:", eval_metrics)

    model.save_pretrained(MODEL_SAVE_DIR)
    tokenizer.save_pretrained(MODEL_SAVE_DIR)

    with open(os.path.join(MODEL_SAVE_DIR, "metrics.json"), "w") as f:
        json.dump({
            "micro_f1": eval_metrics.get("eval_micro_f1"),
            "macro_f1": eval_metrics.get("eval_macro_f1"),
            "precision": eval_metrics.get("eval_precision"),
            "recall": eval_metrics.get("eval_recall"),
            "subset_accuracy": eval_metrics.get("eval_subset_accuracy"),
            "trained_on_rows": actual_sample_size,
            "sample_size": actual_sample_size,
            "full_dataset_size": full_dataset_size,
            "sample_fraction": sample_fraction,
            "max_len": MAX_LEN,
            "token_length_percentiles": length_percentiles,
            "epochs": EPOCHS,
            "data_path": DATA_PATH,
        }, f, indent=2)

    mins = (time.time() - start) / 60
    print(f"\nDone in {mins:.1f} minutes. Model + metrics saved to {MODEL_SAVE_DIR}")
    print("Now run evaluate.py, then app.py.")


if __name__ == "__main__":
    main()

Using device: cpu
Loaded 207814 usable rows


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training on 15000/207814 rows (7.2% of the full dataset) — a compute tradeoff, documented in metrics.json as a limitation.
Token length percentiles: {50: 18, 90: 31, 95: 33, 99: 37} | max observed: 63
Chosen MAX_LEN = 33 (95th percentile, clipped to [16, 64])
Rarest labels (highest weight given): ['amusement', 'anger', 'confusion', 'caring', 'curiosity']


C:\Users\Mahalakshmi\anaconda3\envs\nlp_env\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Precision,Recall,Subset Accuracy
1,0.675000,0.692701,0.305688,0.239316,0.203692,0.612275,0.028889
2,0.624400,0.647006,0.316582,0.270014,0.210165,0.641308,0.043556
3,0.622700,0.639192,0.319402,0.267736,0.213230,0.636163,0.041333


C:\Users\Mahalakshmi\anaconda3\envs\nlp_env\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.

C:\Users\Mahalakshmi\anaconda3\envs\nlp_env\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.

C:\Users\Mahalakshmi\anaconda3\envs\nlp_env\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.




Final validation metrics: {'eval_loss': 0.6391916275024414, 'eval_micro_f1': 0.3194021588707445, 'eval_macro_f1': 0.2677355846750345, 'eval_precision': 0.21322985957132298, 'eval_recall': 0.6361631753031973, 'eval_subset_accuracy': 0.04133333333333333, 'eval_runtime': 45.652, 'eval_samples_per_second': 49.286, 'eval_steps_per_second': 0.789, 'epoch': 3.0}

Done in 49.8 minutes. Model + metrics saved to ./goemotions_model_v3
Now run evaluate.py, then app.py.
